# Cell 1: get the repo and run the unit tests on Kaggle’s library versions


In [1]:
import os, subprocess, sys

REPO_URL = "https://github.com/sarrazer24/evolutionary-prompt-optimization.git"
REPO_DIR = "/kaggle/working/repo"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(subprocess.run(["git", "log", "-3", "--oneline"], capture_output=True, text=True).stdout)
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_llm_interface.py", "tests/test_prompts.py"],
                   capture_output=True, text=True)
print(r.stdout[-600:] or r.stderr[-600:])

Cloning into '/kaggle/working/repo'...


dc42051 Repair missing output rule, tighten mutation similarity, explain rejections

..................................................                       [100%]
50 passed in 9.91s



# Cell 2: load the model and the fixed fitness sample

In [2]:
import json, logging, time
import pandas as pd
import torch

logging.basicConfig(level=logging.WARNING)

from src.llm import LLMInterface, ModelConfig
from src.data.sst2_data import load_saved_splits, stratified_sample

t0 = time.time()
llm = LLMInterface(ModelConfig()).load_model()
print(f"model loaded in {time.time() - t0:.0f}s")
print(json.dumps(llm.info(), indent=2))
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GiB" if torch.cuda.is_available() else "no CUDA")

splits = load_saved_splits("data/processed")          # Member 1's committed opt/val CSVs
N = 1000
sample = stratified_sample(splits.opt, N, seed=42)    # fixed sample from OPT only (never val/test)
print("opt:", len(splits.opt), "| val:", len(splits.val), "| fitness sample:", len(sample),
      "| positive ratio:", round(sample.label.mean(), 3))

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

model loaded in 47s
{
  "model_id": "Qwen/Qwen2.5-1.5B-Instruct",
  "requested_revision": null,
  "resolved_revision": null,
  "dtype": "float16",
  "device": "cuda:0",
  "gpu": "Tesla T4",
  "torch": "2.11.0+cu130",
  "transformers": "5.20.2",
  "inference_config": {
    "decoding": "greedy",
    "max_new_tokens": 8,
    "repetition_penalty": 1.0,
    "dtype": "float16",
    "user_template": "review-sentiment-v1"
  }
}
GPU memory allocated: 2.88 GiB
opt: 53871 | val: 13468 | fitness sample: 1000 | positive ratio: 0.558


# Cell 3: sanity check, speed, memory, and batch-size sensitivity

In [3]:
from src.prompts.baseline_prompts import get_baseline_prompts

baselines = get_baseline_prompts()
demo = sample.head(6)
raw = llm.predict_raw_batch(baselines[0][1], demo.text.tolist())
for text, label, r in zip(demo.text, demo.label, raw):
    print(f"true={'positive' if label else 'negative':8} raw={r!r:14} text={text[:70]!r}")

# --- speed, memory and batch-size sensitivity (same 128 texts, same prompt) ---
texts = sample.text.tolist()[:128]
prompt = baselines[0][1]
outputs, rows = {}, []
for bs in (8, 32, 64):
    llm.config.batch_size = bs
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    t = time.time()
    outputs[bs] = llm.predict_raw_batch(prompt, texts)
    rows.append({"batch_size": bs, "seconds": round(time.time() - t, 1),
                 "texts_per_sec": round(len(texts) / (time.time() - t), 1),
                 "peak_gpu_gib": round(torch.cuda.max_memory_allocated() / 1024**3, 2) if torch.cuda.is_available() else 0})
llm.config.batch_size = 32   # the setting used for every experiment from now on
print(pd.DataFrame(rows).to_string(index=False))
for bs in (8, 64):
    diff = sum(a != b for a, b in zip(outputs[bs], outputs[32]))
    print(f"batch {bs} vs batch 32: {diff}/{len(texts)} different raw outputs")
again = llm.predict_raw_batch(prompt, texts)
print("same batch size, run twice -> identical:", again == outputs[32])

true=negative raw='negative'     text="the film 's mid-to-low budget is betrayed by the surprisingly shoddy m"
true=positive raw='positive'     text='enables shafer to navigate spaces both large ... and small ... with co'
true=positive raw='positive'     text='takes chances that are bold by studio standards'
true=positive raw='positive'     text='director claude chabrol has become the master of innuendo .'
true=negative raw='positive'     text="it 'll probably be in video stores by christmas"
true=negative raw='negative'     text='plumbs uncharted depths of stupidity , incoherence and sub-sophomoric '
 batch_size  seconds  texts_per_sec  peak_gpu_gib
          8      1.8           71.2          2.95
         32      1.4           92.6          3.14
         64      1.5           88.2          3.35
batch 8 vs batch 32: 0/128 different raw outputs
batch 64 vs batch 32: 0/128 different raw outputs
same batch size, run twice -> identical: True


# Cell 4: score all 8 baselines with Member 1’s evaluate_prompt

In [4]:
from src.prompts.baseline_prompts import get_baseline_prompts
baselines = get_baseline_prompts()
from src.evaluation.evaluation import PredictionCache, evaluate_prompt, results_to_dataframe, save_results

cache = PredictionCache("/kaggle/working/outputs/pred_cache.jsonl")

def fitness_eval(prompt, split_name="opt"):
    return evaluate_prompt(prompt, sample.text, sample.label, llm.predict_raw_batch,
                           model_id=llm.model_id, inference_config=llm.inference_config,
                           cache=cache, batch_size=100, split_name=split_name)

results, timings = [], []
for name, prompt in baselines:
    t = time.time()
    results.append(fitness_eval(prompt))
    timings.append(round(time.time() - t, 1))

table = results_to_dataframe(results)
table.insert(0, "name", [n for n, _ in baselines])
table["seconds"] = timings
cols = ["name", "f1_macro", "accuracy", "n_invalid", "invalid_rate", "n_cached", "seconds"]
print(table[cols].round(4).to_string(index=False))

# cache check: the SAME prompt again must cost nothing
t = time.time()
rerun = fitness_eval(baselines[0][1])
print(f"\nre-evaluating baseline_direct: {rerun['n_cached']}/{rerun['n_examples']} from cache, "
      f"{time.time() - t:.2f}s, f1 identical: {rerun['metrics']['f1_macro'] == results[0]['metrics']['f1_macro']}")

# what does the model actually say when it is 'invalid'?
for name, r in zip([n for n, _ in baselines], results):
    bad = [x for x, p in zip(r["raw_responses"], r["y_pred"]) if p == -1][:3]
    if bad:
        print(f"{name}: invalid examples -> {bad}")

save_results(results, "/kaggle/working/outputs/baseline_scores_opt300.csv", append=False)

              name  f1_macro  accuracy  n_invalid  invalid_rate  n_cached  seconds
   baseline_direct    0.9187     0.919          0         0.000         0     11.3
   baseline_expert    0.8940     0.890          9         0.009         0     13.6
 baseline_criteria    0.9364     0.937          0         0.000         0     14.6
 baseline_negation    0.8774     0.877          1         0.001         0     18.7
baseline_recommend    0.9118     0.912          0         0.000         0     16.2
 baseline_fragment    0.9156     0.916          0         0.000         0     17.0
  baseline_fewshot    0.9211     0.918          7         0.007         0     20.8
    baseline_rules    0.9206     0.921          0         0.000         0     18.5

re-evaluating baseline_direct: 1000/1000 from cache, 0.01s, f1 identical: True
baseline_expert: invalid examples -> ['mixed', 'critical', 'mixed']
baseline_negation: invalid examples -> ['neutral']
baseline_fewshot: invalid examples -> ['neutral', 'neu

# Cell 5: real initial population, mutation and crossover

In [5]:
import random
from src.prompts.initial_population import generate_initial_prompts
from src.prompts.operators import mutate_prompt, crossover_prompts
from src.prompts.baseline_prompts import get_baseline_prompt_texts

llm.set_seed(42)
rng = random.Random(42)
seeds = get_baseline_prompt_texts()

calls0 = llm.stats["generate_calls"]
population = generate_initial_prompts(llm, 12, seed_prompts=seeds[:4], rng=rng)
print(f"initial population: {len(population)} prompts, {llm.stats['generate_calls'] - calls0} LLM calls "
      f"(4 are seeds)\n")
for i, p in enumerate(population[4:], 5):
    print(f"[{i}] {p}\n")

mutants, calls0 = [], llm.stats["generate_calls"]
for i in range(6):
    parent = seeds[i % len(seeds)]
    child = mutate_prompt(llm, parent, rng=rng)
    mutants.append((parent, child))
print(f"mutation: 6 requests, {llm.stats['generate_calls'] - calls0} LLM calls, "
      f"{sum(c == p for p, c in mutants)} fell back to the parent")
for parent, child in mutants[:3]:
    print(f"\nPARENT: {parent}\nCHILD : {child}")

children, calls0 = [], llm.stats["generate_calls"]
for i in range(4):
    a, b = rng.sample(seeds, 2)
    children.append((a, b, crossover_prompts(llm, a, b)))
print(f"\ncrossover: 4 requests, {llm.stats['generate_calls'] - calls0} LLM calls, "
      f"{sum(c == a for a, b, c in children)} fell back to parent1")
for a, b, c in children[:2]:
    print(f"\nPARENT A: {a}\nPARENT B: {b}\nCHILD   : {c}")

initial population: 12 prompts, 10 LLM calls (4 are seeds)

[5] Given a movie review text, classify it as positive or negative. Answer with exactly one word: positive or negative.

[6] Please label each movie review as positive or negative, ignoring negation, sarcasm, and irony. Answer with exactly one word: positive or negative.

[7] Please classify the sentiment of the following movie review as positive or negative. Answer with one word: positive or negative.

[8] Decide if the text is positive or negative. Answer with **positive** or **negative**. Answer with exactly one word: positive or negative.

[9] Please classify the following text as positive or negative: Answer with exactly one word: positive or negative.

[10] Did the review express a positive or negative sentiment? Answer with exactly one word: positive or negative.

[11] Is the sentiment of the following movie review positive or negative? Answer with exactly one word: positive or negative.

[12] Identify the overall senti

# Cell 6: score every generated prompt with the same evaluator

In [6]:
candidates = ([("generated", p) for p in population[4:]]
              + [("mutant", c) for p, c in mutants if c != p]
              + [("crossover", c) for a, b, c in children if c != a])
rows = []
for kind, prompt in candidates:
    t = time.time()
    r = fitness_eval(prompt)
    rows.append({"kind": kind, "f1_macro": r["metrics"]["f1_macro"], "accuracy": r["metrics"]["accuracy"],
                 "n_invalid": r["metrics"]["n_invalid"], "seconds": round(time.time() - t, 1),
                 "prompt": prompt[:90]})
cand = pd.DataFrame(rows).sort_values("f1_macro", ascending=False)
print(cand.round(4).to_string(index=False))
print("\nbest baseline f1 :", round(table.f1_macro.max(), 4), "| best generated/evolved f1:", round(cand.f1_macro.max(), 4))
print("\nLLM stats:", llm.stats)
print("cache entries:", len(cache))
cand.to_csv("/kaggle/working/outputs/generated_candidates_opt300.csv", index=False)

     kind  f1_macro  accuracy  n_invalid  seconds                                                                                     prompt
generated    0.9433     0.944          0     14.3 Decide if the text is positive or negative. Answer with **positive** or **negative**. Answ
generated    0.9356     0.936          0     12.5 Please classify the following text as positive or negative: Answer with exactly one word: 
   mutant    0.9336     0.934          0     13.1 Is the review positive or negative? Answer with positive or negative. Answer with exactly 
generated    0.9316     0.932          0     12.9 Please classify the sentiment of the following movie review as positive or negative. Answe
generated    0.9292     0.929          1     12.5 Did the review express a positive or negative sentiment? Answer with exactly one word: pos
crossover    0.9267     0.927          0     20.5 Read the whole text. Decide whether its overall sentiment is positive or negative; if opin
   mutant    